In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fstreet_view_insights%2Fnotebooks%2FIdentify_lamp_make_model_from_RAG%2Futility_pole_analyze_lamp_model.ipynb?utm_source=cropped_street_view_insights_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
</table>

# Identify lamp make and model using Gemini 4 Hydrogen and a RAG

This notebook demonstrates how to identify the make and model of a lamp using the Gemini 4 Hydrogen (`gemini-4-hydrogen`) model with a Retrieval Augmented Generation (RAG) approach.

**Notebook Description:**

The primary goal of this notebook is to leverage the multimodal capabilities of Gemini 4 Hydrogen to analyze images of lamps and extract specific information about their make and model using a Vertex AI RAG Store:

1. **Image Input:** Query Street View Insights observations from BigQuery (`cropped_observations_latest`).
2. **Information Retrieval (RAG):** Retrieve candidate lamp specifications from your Vertex AI RAG Corpus.
3. **Gemini 4 Hydrogen Integration:** Process the input image and retrieved catalog entries with `gemini-4-hydrogen` on the `global` Vertex AI endpoint to identify the most probable make and model.
4. **Output:** Structured lamp post make and model table.

**Prerequisites for First-Time Users:**

1. **Google Cloud Project:** Active Google Cloud project with the Vertex AI API and BigQuery API enabled.
2. **Authentication:** Run `auth.authenticate_user()` in Google Colab or `gcloud auth application-default login` locally (prefer Application Default Credentials or Workload Identity Federation rather than downloaded service account JSON keys).
3. **Input Data:** Access to `imagery_insights___us.cropped_observations_latest` and your own Vertex AI RAG Corpus (`projects/YOUR_PROJECT_ID/locations/us-east4/ragCorpora/YOUR_CORPUS_ID`).

## Install Required Libraries

In [ ]:
!pip install --upgrade google-cloud-bigquery google-genai

## Configuration

**Important**: Replace the placeholder values below with your actual GCP Project ID and Region.

In [ ]:
PROJECT_ID = 'YOUR_PROJECT_ID'  # @param {type:"string"}
GEMINI_LOCATION = 'global'      # @param {type:"string"}
MODEL = 'gemini-4-hydrogen'     # @param {type:"string"}
BIGQUERY_DATASET = 'imagery_insights___us' # @param {type:"string"}
BIGQUERY_TABLE = 'cropped_observations_latest' # @param {type:"string"}
ASSET_TYPE = 'ASSET_CLASS_UTILITY_POLE' # @param {type:"string"}
LIMIT = 100 # @param {type:"integer"}

In [ ]:
# Import necessary libraries
from google.cloud import bigquery
from google import genai
from google.genai import types
from google.genai.types import Part, Tool, Retrieval, VertexRagStore, VertexRagStoreRagResource

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=GEMINI_LOCATION)

print("Libraries imported and google.genai Client initialized on global endpoint.")

## Fetch Image URIs from BigQuery

Next, we'll query a BigQuery table to get the GCS URIs of the images we want to classify.

In [ ]:
# Updated SQL to explicitly select the required columns
BIGQUERY_SQL_QUERY = f"""
SELECT
  asset_id,
  observation_id,
  gcs_uri
FROM
  `{PROJECT_ID}.{BIGQUERY_DATASET}.{BIGQUERY_TABLE}`
WHERE
  asset_type = "{ASSET_TYPE}"
  AND gcs_uri IS NOT NULL
LIMIT {LIMIT};
"""

# Execute BigQuery Query and store the full row data
try:
    bigquery_client = bigquery.Client(project=PROJECT_ID)
    query_job = bigquery_client.query(BIGQUERY_SQL_QUERY)
    # This new variable will hold all the data needed for the DataFrame
    image_data_from_bq = [dict(row) for row in query_job]

    # For compatibility with the existing loop, we also extract the URIs
    gcs_uris = [item.get("gcs_uri") for item in image_data_from_bq if item.get("gcs_uri")]

    print(f"Successfully fetched {len(image_data_from_bq)} items from BigQuery:")
    for item in image_data_from_bq:
        print(f"  - {item}")
except Exception as e:
    print(f"An error occurred while querying BigQuery: {e}")
    image_data_from_bq = []
    gcs_uris = []

## Define Image Classification Function

This function takes a GCS URI and a prompt, then uses the Gemini 4 Hydrogen (`gemini-4-hydrogen`) model to generate a description of the image.

In [ ]:
def classify_image_with_gemini(gcs_uri: str, prompt: str, model_name: str = MODEL) -> str:
    """
    Classifies an image using Gemini 4 Hydrogen by directly passing its GCS URI.
    """
    try:
        image_part = Part.from_uri(file_uri=gcs_uri, mime_type="image/jpeg")
        response = genai_client.models.generate_content(
            model=model_name,
            contents=[image_part, prompt],
        )
        return response.text
    except Exception as e:
        print(f"Error classifying image from URI {gcs_uri}: {e}")
        return "Classification failed."

## Setup for RAG

### Subtask:
Import the necessary libraries for Vertex AI Search and configure the RAG datastore.

In [ ]:
# Define the path to your Vertex AI RAG Corpus
RAG_CORPUS_PATH = "projects/YOUR_PROJECT_ID/locations/us-east4/ragCorpora/YOUR_RAG_CORPUS_ID"  # @param {type:"string"}

print("Configuration updated for Vertex AI RAG.")

## Create a Lamp Post Check Function

### Subtask:
Develop a function that takes an image URI and uses a simple prompt to quickly determine if the image contains a lamp post.

In [ ]:
def is_lamp_post(gcs_uri: str, model_name: str = MODEL) -> bool:
    """
    Uses Gemini 4 Hydrogen to quickly check if an image contains a lamp post.
    """
    try:
        image_part = Part.from_uri(file_uri=gcs_uri, mime_type="image/jpeg")
        prompt = "Does this image contain a lamp post or a street light? Answer with only 'yes' or 'no'."
        response = genai_client.models.generate_content(
            model=model_name,
            contents=[image_part, prompt],
        )
        return (response.text or "").strip().lower() == "yes"
    except Exception as e:
        print(f"Error checking image {gcs_uri}: {e}")
        return False

## Create a Detailed Description Function

This cell defines `get_detailed_description`, which takes a Google Cloud Storage (GCS) URI and uses `gemini-4-hydrogen` on Vertex AI to extract structured visual attributes of the lamp post.

In [ ]:
def get_detailed_description(gcs_uri: str, model_name: str = MODEL) -> str:
    """
    Uses Gemini 4 Hydrogen to generate a detailed description of a lamp post from an image.
    """
    try:
        image_part = Part.from_uri(file_uri=gcs_uri, mime_type="image/jpeg")
        prompt = """
        Describe the lamp post in this image in detail. Focus on the following features:
        - **Overall style**: (e.g., modern, vintage, ornate, simple)
        - **Pole material and color**: (e.g., black metal, grey concrete, brown wood)
        - **Lamp head shape and design**: (e.g., lantern-style, cobra head, globe, multi-light fixture)
        - **Light source**: (e.g., LED, bulb type if visible)
        - **Any distinctive features**: (e.g., decorative elements, banners, bases, arms)
        Provide a concise, structured summary of these features.
        """
        response = genai_client.models.generate_content(
            model=model_name,
            contents=[image_part, prompt],
        )
        return response.text
    except Exception as e:
        print(f"Error generating description for {gcs_uri}: {e}")
        return "Description generation failed."

## Image Classification and Description with Gemini 4 Hydrogen

This section leverages `gemini-4-hydrogen` for two-stage visual inspection and RAG matching:

1. `is_lamp_post`: Quickly checks whether an image contains a lamp post or street light.
2. `get_detailed_description`: Generates a structured visual description of the lamp fixture, pole material, and head geometry for RAG lookup.

In [ ]:
def get_make_and_model(client: genai.Client, description: str, model_name: str = MODEL) -> str:
    """
    Uses the RAG-grounded Gemini 4 Hydrogen model to identify the make and model of a lamp post
    based on its detailed description, using the google.genai SDK.
    """
    try:
        prompt = f"""You are an expert asset inspector. Your task is to find the **closest possible match** for the lamp post described below using the provided dataset.
        Do not give up easily. It is crucial that you find a match if one exists, even if it's not perfect.

        Description to analyze:
        {description}

        Compare this description against the "Description/Key_Features" and "Dimensions" fields in the dataset. Focus on specific details like "250W High Pressure Sodium" or dimensions like '31.5"D x 14.75"W x 14"H'.

        Provide the answer in the format "Make: [Make], Model: [Model]".
        **Only as a last resort**, if there is absolutely no resemblance to any entry, respond with "Make: Unknown, Model: Unknown".
        """

        tools = [
            Tool(
                retrieval=Retrieval(
                    vertex_rag_store=VertexRagStore(
                        rag_resources=[
                            VertexRagStoreRagResource(
                                rag_corpus=RAG_CORPUS_PATH
                            )
                        ]
                    )
                )
            )
        ]

        response = client.models.generate_content(
            model=model_name,
            contents=[prompt],
            config=types.GenerateContentConfig(tools=tools),
        )
        return response.text
    except Exception as e:
        print(f"Error identifying make and model: {e}")
        return "Make and model identification failed."

## Process Images for Lamp Post Detection and Description

This section iterates through a list of image URIs. For each image, it first checks if a lamp post is present using the `is_lamp_post` function. If a lamp post is detected, it then proceeds to generate a detailed description of the lamp post using the `get_detailed_description` function. Finally, it attempts to identify the make and model of the lamp post based on the detailed description using the `get_make_and_model` function, leveraging a RAG-grounded Gemini model. The results of these steps (whether a lamp post was detected, its description, and its identified make/model) are then printed.

In [ ]:
import json

# Reuse PROJECT_ID and GEMINI_LOCATION ("global") configured above
final_results = []
try:
    genai_client = genai.Client(
        vertexai=True, project=PROJECT_ID, location=GEMINI_LOCATION
    )
    print("google.genai client initialized successfully.")
except Exception as e:
    print(f"Error initializing google.genai client: {e}")
    genai_client = None

# Main processing loop to analyze each image
if genai_client:
    for uri in gcs_uris:
        print(f"Processing image: {uri}")

        if is_lamp_post(uri):
            print("  -> Lamp post detected. Generating detailed description...")
            description = get_detailed_description(uri)
            print(f"  -> Generated Description: {description.strip()}")

            print("  -> Identifying make and model using RAG...")
            make_and_model = get_make_and_model(genai_client, description)
            print(f"  -> Identified Make and Model: {make_and_model.strip()}")

            final_results.append({
                "uri": uri,
                "is_lamp_post": True,
                "description": description,
                "make_and_model": make_and_model
            })
        else:
            print("  -> Not a lamp post. Skipping analysis.")
            final_results.append({
                "uri": uri,
                "is_lamp_post": False,
                "description": "N/A",
                "make_and_model": "N/A"
            })

        print("-" * 50)

    print("\n--- Analysis Complete: Final Results ---")
    print(json.dumps(final_results, indent=2))
else:
    print("google.genai client was not initialized. Cannot proceed with analysis.")

## Process Images and Create DataFrame

This section runs the analysis loop, stores the results, and then displays them in a DataFrame.

In [ ]:
import re
import pandas as pd

# Reuse PROJECT_ID and GEMINI_LOCATION ("global") configured above
dataframe_results = []

try:
    genai_client = genai.Client(
        vertexai=True, project=PROJECT_ID, location=GEMINI_LOCATION
    )
    print("google.genai client initialized successfully.")
except Exception as e:
    print(f"Error initializing google.genai client: {e}")
    genai_client = None

if 'genai_client' in locals() and genai_client:
    for item in image_data_from_bq:
        uri = item.get("gcs_uri")
        asset_id = item.get("asset_id")
        observation_id = item.get("observation_id")

        if not uri:
            print(f"Skipping item with missing gcs_uri: {item}")
            continue

        print(f"Processing image for asset {asset_id}: {uri}")

        if is_lamp_post(uri):
            print("  -> Lamp post detected. Generating detailed description...")
            description = get_detailed_description(uri)

            print("  -> Identifying make and model using RAG...")
            make_and_model_str = get_make_and_model(genai_client, description)
            print(f"  -> Identified: {make_and_model_str.strip()}")

            match = re.match(r"Make:\s*(.*?),?\s*Model:\s*(.*)", make_and_model_str.strip(), re.IGNORECASE)

            if match:
                make = match.group(1).strip()
                model = match.group(2).strip()

                if make.lower() not in ["unknown", "n/a"] and model.lower() not in ["unknown", "n/a", "generic incandescent fixture", "generic mv cobra head"]:
                    print(f"  -> SUCCESS: Found Make: {make}, Model: {model}")
                    dataframe_results.append({
                        "asset_id": asset_id,
                        "observation_id": observation_id,
                        "make": make,
                        "model": model
                    })
                else:
                    print("  -> INFO: Generic or unknown make/model found. Skipping for DataFrame.")
            else:
                print("  -> INFO: Could not parse make and model from response. Skipping for DataFrame.")
        else:
            print("  -> Not a lamp post. Skipping analysis.")
        print("-" * 50)

    print("\n--- Analysis Complete --- ")
    if dataframe_results:
        results_df = pd.DataFrame(dataframe_results)
        print("\n--- Identified Lamp Posts --- ")
        display(results_df)
    else:
        print("\n--- No specific lamp post models were identified. ---")
else:
    print("google.genai client was not initialized. Cannot proceed with analysis.")

## Save the dataframe to BigQuery

This cell initializes the google.genai client, which is used for interacting with Gemini models.
It takes the Google Cloud Project ID and Region as parameters.
The client is initialized with vertexai=True to use Vertex AI's backend.
Error handling is included in case the client initialization fails.
The initialized client is stored in the 'genai_client' variable.

In [ ]:
def save_dataframe_to_bigquery(df, project_id, dataset_id, table_id):
    """
    Saves a pandas DataFrame to a specified BigQuery table.

    Args:
        df (pd.DataFrame): The DataFrame to save.
        project_id (str): Your Google Cloud project ID.
        dataset_id (str): The BigQuery dataset ID.
        table_id (str): The BigQuery table ID to create or append to.
    """
    if 'df' not in locals() or df.empty:
        print("The DataFrame is empty or does not exist. Nothing to save to BigQuery.")
        return

    try:
        table_ref = f"{project_id}.{dataset_id}.{table_id}"
        print(f"Attempting to save DataFrame to BigQuery table: {table_ref}")
        df.to_gbq(destination_table=table_ref, project_id=project_id, if_exists='append')
        print(f"Successfully saved DataFrame to BigQuery table: {table_ref}")
    except Exception as e:
        print(f"An error occurred while saving to BigQuery: {e}")

# --- Example Usage (edit and uncomment to run) ---
# if 'results_df' in locals() and not results_df.empty:
#     BIGQUERY_DATASET = 'your_dataset'      # @param {type:"string"}
#     BIGQUERY_TABLE = 'identified_lamp_posts'   # @param {type:"string"}
#     save_dataframe_to_bigquery(results_df, PROJECT_ID, BIGQUERY_DATASET, BIGQUERY_TABLE)
# else:
#     print("DataFrame 'results_df' not found or is empty. Skipping BigQuery upload.")